# Experiment 4: CGM + time_since_meal + image (vit, gated)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [3]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+vit (gated)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'vit'

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gated_vit'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_vit_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_vit_val_stability_summary.json'

In [4]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

val_participant_ids = data.load_multimodal_participant_ids('val')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print()
print('number of test participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print()
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]


number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]


In [5]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = MultimodalGRUPredictor(img_feat_dim=768, **{'gru_hidden_size': 256, 'gru_num_layers': 3, 'gru_dropout': 0.2})

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gated_vit' / f'gated_cgm_time_vit_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)

    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    val_results_per_participant = metrics.evaluate_per_participant(
            y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gated_cgm_time_vit_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Validation results — CGM+time+vit (gated) | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.85 | MAE 8.76 | gRMSE 14.32 | Zone A 94.31% | A+B 99.46%
 30 min | RMSE 18.77 | MAE 12.54 | gRMSE 21.65 | Zone A 88.19% | A+B 99.17%
 60 min | RMSE 26.30 | MAE 17.47 | gRMSE 31.42 | Zone A 79.80% | A+B 98.59%
 90 min | RMSE 31.08 | MAE 20.85 | gRMSE 37.95 | Zone A 74.42% | A+B 98.08%
120 min | RMSE 33.82 | MAE 23.07 | gRMSE 41.68 | Zone A 70.21% | A+B 97.66%

--------------------------------------------------------------------------------
Validation results — CGM+time+vit (gated) | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 13.85 | MAE 10.11 | gRMSE 14.75 | Zone A 92.79% | A+B 99.48%
 30 min | RMSE 19.23 | MAE 13.32 | gRMSE 21.29 | Zone A 86.40% | A+B 99.24%
 60 min | RMSE 26.86 | MAE 18.41 | gRMSE 31.27 | Zone A 77.91% | A+B 98.61%
 90 min | RMSE 31.71 | MAE 22.04 | gRMSE 37.83 | Zone A 72.14% | A+B 98.02%
120 min | RMSE 34.54 | MAE 24.59 | gRMSE 41.58 | Zone A 66.81% | A+B 97.68%

--------------------------------------------------------------------------------
Validation results — CGM+time+vit (gated) | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


...

 15 min | RMSE 13.11 | MAE 8.87 | gRMSE 14.73 | Zone A 94.19% | A+B 99.48%
 30 min | RMSE 19.04 | MAE 12.68 | gRMSE 22.00 | Zone A 88.05% | A+B 99.17%
 60 min | RMSE 26.50 | MAE 17.54 | gRMSE 31.64 | Zone A 79.65% | A+B 98.60%
 90 min | RMSE 31.21 | MAE 20.84 | gRMSE 38.12 | Zone A 74.47% | A+B 98.09%
120 min | RMSE 33.92 | MAE 23.06 | gRMSE 41.75 | Zone A 70.55% | A+B 97.62%

--------------------------------------------------------------------------------
Validation results — CGM+time+vit (gated) | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.86 | MAE 8.65 | gRMSE 14.07 | Zone A 94.37% | A+B 99.50%
 30 min | RMSE 18.74 | MAE 12.44 | gRMSE 21.31 | Zone A 88.35% | A+B 99.19%
 60 min | RMSE 26.48 | MAE 17.39 | gRMSE 31.59 | Zone A 79.93% | A+B 98.63%
 90 min | RMSE 31.28 | MAE 20.74 | gRMSE 38.36 | Zone A 74.71% | A+B 98.04%
120 min | RMSE 34.05 | MAE 23.01 | gRMSE 42.11 | Zone A 70.85% | A+B 97.68%

--------------------------------------------------------------------------------
Validation results — CGM+time+vit (gated) | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.83 | MAE 8.66 | gRMSE 14.26 | Zone A 94.66% | A+B 99.50%
 30 min | RMSE 18.78 | MAE 12.40 | gRMSE 21.48 | Zone A 88.74% | A+B 99.19%
 60 min | RMSE 26.58 | MAE 17.31 | gRMSE 31.77 | Zone A 80.25% | A+B 98.56%
 90 min | RMSE 31.50 | MAE 20.74 | gRMSE 38.44 | Zone A 74.89% | A+B 98.06%
120 min | RMSE 34.03 | MAE 22.90 | gRMSE 41.88 | Zone A 71.09% | A+B 97.63%

--------------------------------------------------------------------------------


In [7]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [8]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [9]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== CGM+time+vit (gated) validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 13.01 +/- 0.31 (2.79) mg/dL   MAE = 8.88 +/- 0.45 (1.89)   gRMSE = 14.34 +/- 0.26 (3.85)   Zone A = 94.31 +/- 0.57 (5.43)%   Zone A+B = 99.48 +/- 0.02 (0.95)%
 30-min:  RMSE = 18.85 +/- 0.16 (4.30) mg/dL   MAE = 12.58 +/- 0.28 (2.79)   gRMSE = 21.46 +/- 0.25 (6.49)   Zone A = 88.14 +/- 0.66 (6.06)%   Zone A+B = 99.19 +/- 0.02 (1.15)%
 60-min:  RMSE = 26.55 +/- 0.16 (7.99) mg/dL   MAE = 17.50 +/- 0.34 (5.24)   gRMSE = 31.60 +/- 0.19 (12.29)   Zone A = 79.76 +/- 0.71 (6.79)%   Zone A+B = 98.59 +/- 0.02 (1.52)%
 90-min:  RMSE = 31.40 +/- 0.20 (11.13) mg/dL   MAE = 20.89 +/- 0.44 (7.18)   gRMSE = 38.27 +/- 0.27 (16.96)   Zone A = 74.47 +/- 0.90 (8.63)%   Zone A+B = 98.06 +/- 0.02 (2.13)%
120-min:  RMSE = 34.10 +/- 0.21 (12.61) mg/dL   MAE = 23.17 +/- 0.51 (8.03)   gRMSE = 41.91 +/- 0.25 (19.33)   Zone A = 70.39 +/- 1.31 (9.09)%   Zone A+B = 97.67 +/- 0.03 (2.73)%


In [10]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_vit/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_vit/gated_cgm_time_vit_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_vit/gated_cgm_time_vit_stability_summary.json
